<img src=https://upload.wikimedia.org/wikipedia/commons/6/68/Logo_universidad_icesi.svg width=300>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebastianb92/nlp-labs/blob/main/Session4/1-Text-classification-BERT.ipynb)


# Maestría en Inteligencia Artificial  
## Procesamiento de Lenguaje natural
### Sesión 4 - Práctica

---


**Integrantes:**  
- Johan Sebastian Bonilla  
- Edwin Gómez  

# Introducción

# Clasificación de textos - BERT

En este notebook se aborda el problema de clasificación de texto aplicado a la detección de contenido relacionado con el cambio climático. Para ello, se utiliza el dataset somosnlp/spa_climate_detection, el cual contiene textos en español etiquetados según su relación con esta temática.

El objetivo principal es desarrollar un modelo capaz de clasificar automáticamente textos en función de si están relacionados o no con el cambio climático. Para lograrlo, se emplea el modelo BERT, una arquitectura basada en transformers que permite capturar el contexto semántico del lenguaje de manera bidireccional, mejorando significativamente el rendimiento en tareas de procesamiento de lenguaje natural (NLP).

# 1. Configurar entorno

En esta sección se configuran las librerías y dependencias necesarias para el análisis de datos y procesamiento de lenguaje natural. Esto garantiza que el entorno esté listo para cargar, limpiar y analizar las reseñas.

In [ ]:
import importlib.metadata
import warnings

warnings.filterwarnings('ignore')

installed_packages = [dist.metadata['Name'].lower() for dist in importlib.metadata.distributions()]
IN_COLAB = 'google-colab' in installed_packages

In [ ]:
!pip install torchinfo > /dev/null 2>&1
!pip install evaluate > /dev/null 2>&1

# 2. Recopilación de datos

Para el presente análisis se utilizará el conjunto de datos somosnlp/spa_climate_detection, disponible en Hugging Face, el cual contiene textos en español relacionados con la temática del cambio climático. Cada registro del dataset se encuentra etiquetado según si el contenido está o no asociado a este tema.

Este conjunto de datos permitirá evaluar el desempeño de modelos de procesamiento de lenguaje natural en una tarea de clasificación binaria de texto. En particular, se empleará el modelo BERT, el cual será ajustado (fine-tuning) sobre este corpus para aprovechar su capacidad de comprensión contextual del lenguaje.

De esta manera, se busca construir un modelo capaz de identificar automáticamente si un texto está relacionado con el cambio climático, evaluando su rendimiento mediante métricas estándar de clasificación.

In [ ]:
from datasets import load_dataset
import warnings
import os

warnings.filterwarnings("ignore")
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
dataset = load_dataset('somosnlp/spa_climate_detection', split='train')
dataset

In [ ]:
dataset[1]

# 3. Analisis Exploratorio

En esta sección se realiza un análisis exploratorio del conjunto de datos somosnlp/spa_climate_detection con el objetivo de comprender su estructura, distribución y características principales antes del proceso de modelado.

Inicialmente, se examina la cantidad total de registros, así como la distribución de las clases, con el fin de identificar posibles problemas de desbalanceo que puedan afectar el desempeño del modelo. Asimismo, se analizan características básicas de los textos, como la longitud de las secuencias, la frecuencia de palabras y la presencia de términos relevantes asociados al cambio climático.

Adicionalmente, se exploran ejemplos representativos de cada clase para entender mejor el tipo de contenido presente en el dataset y la complejidad de la tarea de clasificación. Este análisis permite identificar patrones, posibles ruidos en los datos y definir estrategias adecuadas de preprocesamiento y modelado.



## Estructura del Dataset

In [ ]:
df = dataset.to_pandas()
df.head()

Revisamos los primero 5 registros del dataset para validar estructura y valores

## Análisis de Distribución

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

labels = dataset["answer"]

# Contar clases
counter = Counter(labels)

# Mostrar conteo
print("Distribución absoluta:")
print(counter)

# Calcular proporciones
total = sum(counter.values())
print("\nDistribución porcentual:")
for cls, freq in counter.items():
    print(f"Clase {cls}: {freq} ({freq/total:.2%})")


plt.figure()
bars = plt.bar(counter.keys(), counter.values())

plt.xticks(list(counter.keys()))
plt.xlabel("Clase (0 = No clima, 1 = Clima)")
plt.ylabel("Número de registros")
plt.title("Distribución de clases en el dataset")

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height,
             f'{height}', ha='center', va='bottom')

plt.show()

La distribución de clases en el dataset es relativamente equilibrada, con una ligera predominancia de la clase 1 (Texto relacionado con cambio climatico) (55.17%) frente a la clase 0 (Texto no relacionado con cambio climatico) (44.83%).

Esto sugiere que no existe un desbalance significativo que pueda afectar gravemente el entrenamiento del modelo.


## Análisis del Texto

In [ ]:
import numpy as np

text_lengths = [len(row['question'].split()) for row in dataset]

print(f"Texto más corto: {min(text_lengths)}")
print(f"Texto más largo: {max(text_lengths)}")
print(f"Longitud promedio: {sum(text_lengths) / len(text_lengths)}")

print("\nPercentiles:")
for p in [50, 75, 90, 95, 99]:
    print(f"{p}%: {np.percentile(text_lengths, p)}")

Los textos del dataset presentan una alta variabilidad en su longitud, con valores que van desde 2 hasta 604 palabras.

La longitud promedio es de aproximadamente 114 palabras, aunque la mediana (50%) se sitúa en 59, lo que indica una distribución sesgada con presencia de textos considerablemente más largos.

 Esto se confirma con los percentiles superiores, donde un pequeño porcentaje de textos alcanza longitudes significativamente mayores.

In [ ]:
df['Palabras por clase'] = df['question'].str.split().apply(len)
df.groupby('answer')['Palabras por clase'].median()

In [ ]:
df.boxplot(column='Palabras por clase', by='answer', grid=False)
plt.title('Distribución de longitud de textos por clase')
plt.suptitle('')
plt.xlabel('Clase (0 = No clima, 1 = Clima)')
plt.ylabel('Número de palabras')
plt.show()

Se evidencia diferencias en la distribución de la longitud de los textos entre las clases. La clase 0 presenta una mayor dispersión y valores más extremos, indicando la presencia de textos considerablemente más largos, mientras que la clase 1 tiene una distribución más concentrada y homogénea.

Esto sugiere que la longitud del texto podría variar según la clase, aunque con cierto solapamiento entre ambas.

## Vocabulario

In [ ]:
from collections import Counter

all_words = []
for text in dataset["question"]:
    all_words.extend(text.split())

vocab = Counter(all_words)
print("Tamaño del vocabulario:", len(vocab))

El dataset cuenta con 41136 palabras unicas lo que indica una alta diversidad léxica en los textos.

## Análisis de Ruido

In [ ]:
df.sample(5)

Podemos ver aleatoriamente algunos registros para rectificar la clasificacion o valor de la etiqueta vs el texto

## Palabras más frecuentes y N-Gramas por clase

Analizamos las palabras y bigramas más frecuentes en cada polaridad, excluyendo stopwords, para identificar el vocabulario diferenciador entre comentarios de cada clase.

In [ ]:
from collections import Counter
import nltk
from nltk.corpus import stopwords
import matplotlib.pyplot as plt
nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("spanish"))
extra_stopwords = {
    'de', 'la', 'el', 'en', 'y', 'a', 'los', 'las', 'del', 'se', '(', ')',
    'que', 'con', 'un', 'una', 'es', 'no', 'lo', 'su', 'por', 'al', ',', '.', 'más', 'le', 'me', 'mi'
}
stop_words.update(extra_stopwords)

def top_words_by_class(ds, label, n=15, sw=None):
    words = []
    for row in ds:
        if row['answer'] == label:
            words.extend(row['question'].lower().split())
    if sw:
        words = [w for w in words if w not in sw]
    return Counter(words).most_common(n)

def get_bigrams(text):
    tokens = text.lower().split()
    return [f"{tokens[i]} {tokens[i+1]}" for i in range(len(tokens) - 1)]

top_pos = top_words_by_class(dataset, 1, sw=stop_words)
top_neg = top_words_by_class(dataset, 0, sw=stop_words)

bg_pos, bg_neg = Counter(), Counter()
for row in dataset:
    bgs = get_bigrams(row['question'])
    if row['answer'] == 1:
        bg_pos.update(bgs)
    else:
        bg_neg.update(bgs)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, top, title, color in [
    (axes[0, 0], top_pos,                'Top 15 palabras — Relacionado', 'steelblue'),
    (axes[0, 1], top_neg,                'Top 15 palabras — No relacionado',  'salmon'),
    (axes[1, 0], bg_pos.most_common(10), 'Top 10 bigramas — Relacionado', 'steelblue'),
    (axes[1, 1], bg_neg.most_common(10), 'Top 10 bigramas — No relacionado',  'salmon'),
]:
    terms, freqs = zip(*top)
    ax.barh(list(terms)[::-1], list(freqs)[::-1], color=color)
    ax.set_title(title)
    ax.set_xlabel('Frecuencia')

plt.tight_layout()
plt.show()

- **Existe una clara diferenciación léxica entre clases**: en la clase relacionada con cambio climático predominan términos como *cambio, climático, emisiones, calentamiento y carbono*, lo que indica un vocabulario altamente específico del dominio.

- **Clase relacionada (1)** : se observan palabras directamente asociadas al contexto ambiental y científico, lo que sugiere que los textos contienen información más técnica o explícitamente vinculada al cambio climático.

- **Clase no relacionada (0)**: predominan términos más generales o contextuales como *gobierno, madrid, partido, vox*, lo que indica que muchos textos pertenecen a ámbitos políticos o sociales, pero no necesariamente al tema climático.

- **Baja superposición semántica relevante**: a diferencia de otros problemas (como sentimiento), aquí las clases parecen diferenciarse más por el tema que por el tono, lo que facilita la tarea de clasificación.

## Sobre los n-gramas (bigramas)

- Los bigramas más frecuentes (ej. de la, en el, de los) son principalmente estructuras gramaticales comunes del español, por lo que aportan poco valor discriminativo por sí solos.

- En la clase relacionada, aparecen combinaciones más informativas como *cambio climático, millones de, el cambio*, que refuerzan el contexto temático.

- En la clase no relacionada, los bigramas siguen siendo mayormente genéricos y estructurales, lo que confirma la menor presencia de expresiones específicas del dominio climático.

- Para mejorar la capacidad discriminativa, sería útil enfocarse en n-gramas más semánticos o aplicar técnicas que capturen mejor el contexto, como embeddings generados por BERT.


# 4. Tokenizador

En esta etapa se realiza el proceso de tokenización del texto utilizando el tokenizador preentrenado asociado al modelo BERT. En particular, se emplea el tokenizador correspondiente al modelo dccuchile/bert-base-spanish-wwm-cased, disponible en la librería Hugging Face Transformers.

Es importante destacar que, para modelos basados en transformers, se debe utilizar el mismo tokenizador con el que el modelo fue entrenado originalmente. Esto garantiza que la representación de los textos sea consistente con la arquitectura del modelo y permite aprovechar adecuadamente el conocimiento previamente aprendido durante el preentrenamiento.

El tokenizador de BERT divide el texto en sub-palabras (subwords), lo que permite manejar palabras desconocidas y capturar mejor la estructura del lenguaje. Como resultado, cada texto es transformado en una secuencia de identificadores numéricos (tokens), que pueden ser procesados posteriormente por el modelo para la tarea de clasificación.

In [ ]:
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from tokenizers.pre_tokenizers import ByteLevel

model_name = "dccuchile/bert-base-spanish-wwm-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

Probamos el tokenizador obtenido

In [ ]:
tokenizer.pad_token = '[PAD]'
tokenizer("hola mundo!!", max_length=10, truncation=True, padding='max_length').tokens()

Tamaño del vocabulario

In [ ]:
tokenizer.vocab_size

Longitud máxima de tokens que el modelo puede procesar en una sola entrada.

In [ ]:
tokenizer.model_max_length

Revisamos los nombres de las entradas que el modelo espera recibir después del proceso de tokenización

In [ ]:
tokenizer.model_input_names

¿Qué significa cada uno?

- input_ids:
Son los tokens convertidos a números (lo que realmente procesa el modelo)

- attention_mask:
Indica qué tokens son reales (1) y cuáles son padding (0)

- token_type_ids:
Se usan para diferenciar segmentos de texto (por ejemplo, en tareas con dos oraciones)

In [ ]:
tokens = sorted(tokenizer.vocab.items(), key=lambda x: x[1], reverse=False)
print(f"Vocabulario: {tokenizer.vocab_size} tokens")
print("Primeros 15 tokens:")
print([f"{tokenizer.convert_tokens_to_string([t])}" for t, _ in tokens[:15]])
print("15 tokens de en medio:")
print([f"{tokenizer.convert_tokens_to_string([t])}" for t, _ in tokens[20000:20020]])
print("Últimos 15 tokens:")
print([f"{tokenizer.convert_tokens_to_string([t])}" for t, _ in tokens[-15:]])

# 5. Definición del modelo BERT pre-entrenado

En esta etapa se define el modelo preentrenado que será utilizado para la tarea de clasificación de texto. Se emplea el modelo dccuchile/bert-base-spanish-wwm-cased, basado en la arquitectura BERT, el cual ha sido previamente entrenado sobre grandes volúmenes de texto en español.

El uso de un modelo preentrenado permite aprovechar el conocimiento lingüístico adquirido durante su entrenamiento inicial, facilitando la comprensión del contexto y las relaciones semánticas entre palabras. Posteriormente, este modelo es ajustado (fine-tuning) específicamente para la tarea de clasificación de textos relacionados con el cambio climático.

De esta manera, se logra una mejora significativa en el rendimiento del modelo en comparación con enfoques tradicionales, al utilizar representaciones profundas del lenguaje.

In [ ]:
id2category = dict(enumerate(np.unique(df['answer'])))
category2id = {v: k for k, v in id2category.items()}

In [ ]:
import torch
from torchinfo import summary
from transformers import AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
inputs = tokenizer("hola mundo!!!", max_length=10, truncation=True, padding='max_length', return_tensors='pt')

print(f"Input Shapes & Types:")
print({k: (v.shape, v.dtype) for k, v in inputs.items()})

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(category2id)).to(device)

# Congelamos los pesos del modelo base para usarlo como featurizer solamente.
for param in model.base_model.parameters():
    param.requires_grad = False


input_sizes = [inputs['input_ids'].shape] * 3
input_types = [inputs['input_ids'].dtype] * 3
with torch.no_grad():
    print(summary(model, input_size=input_sizes, dtypes=input_types, col_names=['input_size', 'output_size', 'num_params', 'trainable']))

Se utiliza BERT como extractor de características, manteniendo sus pesos congelados y entrenando únicamente la capa de clasificación final

In [ ]:
with torch.no_grad():
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model(**inputs)
print({k: v.shape for k, v in outputs.items()})

La salida del modelo contiene los logits con dimensión [1, 2], donde cada valor representa la puntuación asociada a cada clase en la clasificación binaria

In [ ]:
outputs

In [ ]:
model.classifier

Esta capa lineal transforma la representación generada por BERT (768 dimensiones) en dos valores, correspondientes a las clases del problema de clasificación binaria

### Instanciamos los datasets

Dividimos el dataset en conjuntos de entrenamiento (80%), validación (10%) y prueba (10%).

In [ ]:
training_dataset = dataset.train_test_split(train_size=0.8)
validation_dataset = training_dataset['test'].train_test_split(train_size=0.5)

In [ ]:
from datasets.dataset_dict import DatasetDict

new_dataset = DatasetDict({
    'train': training_dataset['train'],
    'val': validation_dataset['train'],
    'test': validation_dataset['test'],
})
new_dataset

In [ ]:
def preprocess_function(max_len):
    def _preprocess_function(examples):
        return tokenizer(examples['question'], max_length=max_len, truncation=True, padding='max_length')
    return _preprocess_function

def tokenize(max_len: int = 8):
    def _tokenize(batch):
        return tokenizer(batch['question'], max_length=max_len, truncation=True, padding='max_length')
    return _tokenize

def category_names_2_ids(batch):
    batch['label'] = category2id[batch['answer']]
    return batch

Se tokenizan los textos y se convierten las etiquetas a formato numérico, generando un dataset listo para ser utilizado en el entrenamiento del modelo.

In [ ]:
tokenized_dataset = new_dataset.map(preprocess_function(max_len=512), batched=True)
tokenized_dataset = tokenized_dataset.map(category_names_2_ids)
tokenized_dataset

### Definición del proceso de entrenamiento

En esta etapa se configuran los parámetros y el proceso de entrenamiento del modelo utilizando la clase Trainer de la librería Hugging Face Transformers. Este componente facilita la gestión del entrenamiento, evaluación y registro de métricas de manera estructurada.

In [ ]:
from transformers import Trainer, TrainingArguments
from typing import Dict, Any
import evaluate

# Definimos la función métrica de calidad
accuracy = evaluate.load("accuracy")

def compute_metrics(pred) -> Dict[str, Any]:
    """compute metrics

    Esta función será invocada en
    cada epoca y la utilizaremos para
    calcular la métrica de calidad.
    """
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    # Retorna un diccionario como {'nombre-metrica': valor}
    acc = accuracy.compute(predictions=preds, references=labels)
    return acc


batch_size = 8 if IN_COLAB else 4
logging_steps = len(tokenized_dataset['train']) // batch_size
# Definimos los parámetros globales de entrenamiento
training_args = TrainingArguments(
    output_dir='./hf',
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_eval_batch_size=batch_size,
    per_device_train_batch_size=batch_size,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    disable_tqdm=False,
    logging_steps=logging_steps,
    report_to='tensorboard'
)

# Y definimos el entrenador, especificando el modelo, datasets y el tokenizador
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['val'],
    processing_class=tokenizer
)

Ejecutamos el entrenamiento

In [ ]:
%%time
trainer.train()

Revisamos metricas de rendimiento

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir hf/runs

**Resultados del entrenamiento**


- La pérdida de entrenamiento disminuye de 0.678 a 0.642, lo que indica que el modelo está aprendiendo patrones en los datos.

- La pérdida de validación también presenta una ligera reducción, sugiriendo que el modelo generaliza de manera estable.

- La accuracy se mantiene constante en 0.662, lo que indica que, aunque el modelo mejora en términos de pérdida, no logra incrementar su capacidad de clasificación en el conjunto de validación.

Evaluamos en el conjunto de prueba

In [ ]:
model.eval()
trainer.evaluate(tokenized_dataset['test'])

**Evaluacion en el conjunto de prueba**

- La accuracy de 70.3% indica que el modelo tiene una capacidad aceptable para clasificar correctamente los textos, superando el desempeño observado durante la validación (~66%).

- La reducción en la loss frente a validación sugiere que el modelo logra una mejor generalización sobre datos no vistos.

- El rendimiento obtenido es consistente con un modelo que ha capturado patrones relevantes del lenguaje, aunque aún existe margen de mejora.

In [ ]:
predictions = trainer.predict(tokenized_dataset['test'])

In [ ]:
predicted_labels = np.argmax(predictions.predictions, axis=-1)
test_set = tokenized_dataset['test']
test_set = test_set.add_column('prediction_label', predicted_labels)
test_set = test_set.add_column('prediction', list(map(lambda label: id2category[label], predicted_labels)))
test_set

Revisamos las etiquetas reales vs las predichas

In [ ]:
columns = ['question', 'answer',  'prediction']
test_set.set_format('pandas', columns=columns)
df = test_set.to_pandas()[columns]
df.style.set_table_styles(
    [
        {'selector': 'td', 'props': [('word-wrap', 'break-word')]}
    ]
)
df.head(15)

Revisamos donde el modelo no acerto en la prediccion

In [ ]:
errors = df[df['answer'] != df['prediction']]
errors.head(15)

# 6. Uso de clasificador especializado

Hasta este punto, se ha utilizado la configuración estándar de BertForSequenceClassification, la cual emplea una única capa lineal como clasificador final. Si bien esta aproximación es eficiente, puede resultar limitada para capturar relaciones más complejas en los datos.

Con el objetivo de mejorar el desempeño del modelo, se propone implementar un clasificador personalizado, añadiendo mayor profundidad a la arquitectura mediante múltiples capas densas. En este enfoque, el modelo BERT continúa utilizándose como extractor de características (feature extractor), mientras que la nueva cabeza de clasificación introduce mayor capacidad de aprendizaje no lineal.

Cargamos nuevamente el modelo

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(category2id)).to(device)

for param in model.base_model.parameters():
    param.requires_grad = False


input_sizes = [inputs['input_ids'].shape] * 3
input_types = [inputs['input_ids'].dtype] * 3
with torch.no_grad():
    print(summary(model, input_size=input_sizes, dtypes=input_types, col_names=['input_size', 'output_size', 'num_params', 'trainable']))

Rectificamos el clasificador

In [ ]:
model.classifier

El clasificador propuesto está compuesto por:

- Capa Lineal (768 → 256): reducción de dimensionalidad y extracción de características relevantes

- Layer Normalization: mejora la estabilidad del entrenamiento

- ReLU: introduce no linealidad

- Dropout: reduce el riesgo de sobreajuste

- Capa final (256 → 2): genera los logits para la clasificación binaria

In [ ]:
import torch.nn as nn


classifier = nn.Sequential(
    nn.Linear(768, 256),
    nn.LayerNorm(256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 2)
)

# simplemente reemplazamos el clasificador existente por el nuestro:
model.classifier = classifier
with torch.no_grad():
    print(summary(model, input_size=input_sizes, dtypes=input_types, col_names=['input_size', 'output_size', 'num_params', 'trainable']))

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['val'],
    processing_class=tokenizer
)

In [ ]:
%%time

trainer.train()

Tras implementar el clasificador personalizado sobre BERT, se observan mejoras significativas en el desempeño del modelo:

- La pérdida de entrenamiento disminuye de 0.53 a 0.40, evidenciando un aprendizaje efectivo.

- La pérdida de validación también se reduce de 0.40 a 0.37, lo que indica una buena capacidad de generalización.

- La accuracy alcanza un 82.76%, mostrando una mejora considerable frente al modelo base (~66%).

In [ ]:
model.eval()
trainer.evaluate(tokenized_dataset['test'])

- La accuracy del 86.55% evidencia una mejora significativa respecto al modelo base, confirmando la efectividad del clasificador personalizado.

- La baja pérdida (loss) indica que el modelo no solo acierta en sus predicciones, sino que lo hace con alta confianza.

- El desempeño superior en test frente a validación (~82.7%) sugiere una buena capacidad de generalización.

**Fine Tuning con BERT**
Para terminar, ahora harémos fine tuning, es decir, vamos a dejar libres todas las capas del modelo base para que todas calculen gradiente y entrenen sobre nuestra tarea específica.

En este caso entonces no necesitamos modificar nada del modelo original, podemos instanciarlo y proceder directamente al entrenamiento:

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(category2id)).to(device)
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['val'],
    processing_class=tokenizer
)

In [ ]:
%%time
trainer.train()

In [ ]:
model.eval()
trainer.evaluate(tokenized_dataset['test'])

## Curvas de Entrenamiento

Visualizamos la evolución del loss y la accuracy durante el entrenamiento y la validación, leyendo directamente los logs guardados por TensorBoard. Esto permite identificar sobreajuste, convergencia temprana u otros comportamientos durante el entrenamiento.

In [ ]:
import glob
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

log_dir = 'tb_logs/TransformersClassifier'
versions = sorted(glob.glob(f'{log_dir}/version_*'))

if not versions:
    print('No se encontraron logs de entrenamiento en', log_dir)
else:
    ea = EventAccumulator(versions[-1])
    ea.Reload()
    tags = ea.Tags().get('scalars', [])
    print('Métricas disponibles:', tags)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for (train_tag, val_tag, ylabel), ax in zip(
        [('train-loss', 'val-loss', 'Loss'), ('train-acc', 'val-acc', 'Accuracy')],
        axes
    ):
        for tag, label, ls in [(train_tag, 'Train', '-'), (val_tag, 'Validación', '--')]:
            if tag in tags:
                events = ea.Scalars(tag)
                ax.plot([e.step for e in events], [e.value for e in events], linestyle=ls, label=label)
        ax.set_xlabel('Época')
        ax.set_ylabel(ylabel)
        ax.set_title(f'Curva de {ylabel}')
        ax.legend()

    plt.tight_layout()
    plt.show()

El modelo de clasificación de sentimientos basado en Transformer obtuvo una precisión del 85.29% en el conjunto de prueba, lo que significa que el modelo logra clasificar correctamente la mayoría de los comentarios como positivos o negativos. Este resultado muestra que el modelo aprendió a identificar patrones y relaciones entre las palabras de los textos para determinar el tipo de sentimiento expresado en cada comentario.

Aunque el desempeño es bueno, todavía existe un pequeño porcentaje de errores, por lo que el modelo podría mejorarse utilizando más datos de entrenamiento, ajustando algunos parámetros o probando versiones más avanzadas del modelo.

# 10. Hacemos predicciones

In [ ]:
predictions = trainer.predict(model, test_loader)
predictions = torch.cat(predictions, dim=0)
predictions = torch.argmax(predictions, dim=-1)
predictions = [spanish_sentiment_dataset.id_2_class_map[pred] for pred in predictions.numpy()]

In [ ]:
import pandas as pd

# Obtener predicciones
predictions = trainer.predict(model, test_loader)
predictions = torch.cat(predictions, dim=0)
pred_labels = torch.argmax(predictions, dim=-1).numpy()

# Obtener índices del test set
test_indices = test_dataset.indices

# Crear DataFrame con resultados
id_2_token = {v: k for k, v in vocab.items()}

df_results = pd.DataFrame({
    'texto': [dataset[i]['text'] for i in test_indices],
    'etiqueta_real': [dataset[i]['label'] for i in test_indices],
    'prediccion': pred_labels,
    'sentimiento_real': [sentiment_dataset.id_2_class_map[dataset[i]['label']] for i in test_indices],
    'sentimiento_pred': [sentiment_dataset.id_2_class_map[p] for p in pred_labels]
})

df_results['correcto'] = df_results['etiqueta_real'] == df_results['prediccion']
print(f"Precisión: {df_results['correcto'].mean():.4f}")
print(f"Total correctos: {df_results['correcto'].sum()} / {len(df_results)}")
df_results.head(10)

In [ ]:
# Análisis de errores
errors = df_results[~df_results['correcto']]
print(f"\nTotal de errores: {len(errors)}")
print(f"\nEjemplos de predicciones incorrectas:")
errors[['texto', 'sentimiento_real', 'sentimiento_pred']].head(10)

# 11. Analizamos los resultados

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Matriz de confusión
cm = confusion_matrix(df_results['etiqueta_real'], df_results['prediccion'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Matriz de confusión
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[0].text(j + 0.5, i + 0.5, str(cm[i, j]),
                     ha='center', va='center', color='black', fontsize=14, fontweight='bold')

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'], ax=axes[0])

axes[0].set_xlabel('Predicción')
axes[0].set_ylabel('Real')
axes[0].set_title('Matriz de Confusión')

# Plot 2: Distribución de predicciones
pred_counts = df_results['sentimiento_pred'].value_counts()
axes[1].bar(pred_counts.index, pred_counts.values, color=['salmon', 'lightgreen'])
axes[1].set_xlabel('Sentimiento')
axes[1].set_ylabel('Cantidad')
axes[1].set_title('Distribución de Predicciones')

plt.tight_layout()
plt.show()

# Reporte de clasificación
print("\nReporte de Clasificación:")
print(classification_report(df_results['etiqueta_real'], df_results['prediccion'],
                          target_names=['Negative', 'Positive']))

## Curva ROC y Precisión-Recall

In [ ]:
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

model.eval()
all_probs, all_true = [], []

with torch.no_grad():
    for batch_data in test_loader:
        x_b    = batch_data['input_ids']
        mask_b = batch_data['attention_mask']
        y_b    = batch_data['y']
        logits = model(x_b, mask_b)
        probs  = torch.softmax(logits, dim=-1)[:, 1]   # probabilidad clase positiva
        all_probs.extend(probs.cpu().numpy())
        all_true.extend(y_b.cpu().numpy())

fpr, tpr, _        = roc_curve(all_true, all_probs)
roc_auc            = auc(fpr, tpr)
prec, rec, _       = precision_recall_curve(all_true, all_probs)
avg_precision      = average_precision_score(all_true, all_probs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--', label='Clasificador aleatorio')
axes[0].set_xlabel('Tasa de Falsos Positivos')
axes[0].set_ylabel('Tasa de Verdaderos Positivos')
axes[0].set_title('Curva ROC')
axes[0].legend()

axes[1].plot(rec, prec, color='darkorange', lw=2, label=f'PR (AP = {avg_precision:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precisión-Recall')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"AUC-ROC: {roc_auc:.4f} | Average Precision: {avg_precision:.4f}")

Las curvas ROC y Precisión-Recall muestran que el modelo tiene una capacidad **moderada** para separar clases: su desempeño es claramente mejor que el azar, pero aún limitado para distinguir correctamente la clase minoritaria (negativa). Esto es consistente con el desbalance del dataset y con la distribución de predicciones observada, donde el modelo tiende a favorecer la clase positiva. En consecuencia, aunque la precisión global es aceptable, todavía hay margen de mejora en **recall y precisión de la clase negativa**.

## Análisis de Errores por Longitud de Texto

Analizamos si la longitud del texto (número de palabras) influye en el desempeño del modelo, identificando en qué rangos el clasificador comete más errores.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_results['longitud_palabras'] = df_results['texto'].apply(lambda t: len(t.split()))

bins       = [0, 5, 10, 20, 40, 200]
bin_labels = ['1-5', '6-10', '11-20', '21-40', '>40']
df_results['rango_longitud'] = pd.cut(
    df_results['longitud_palabras'], bins=bins, labels=bin_labels, right=True
)

acc_by_len = (
    df_results.groupby('rango_longitud', observed=True)['correcto']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'accuracy', 'count': 'n_muestras'})
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(acc_by_len.index.astype(str), acc_by_len['accuracy'], color='steelblue')
axes[0].axhline(
    df_results['correcto'].mean(), color='red', linestyle='--',
    label=f"Media global ({df_results['correcto'].mean():.2f})"
)
axes[0].set_xlabel('Rango de longitud (palabras)')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy por longitud de texto')
axes[0].set_ylim(0, 1)
axes[0].legend()

axes[1].bar(acc_by_len.index.astype(str), acc_by_len['n_muestras'], color='coral')
axes[1].set_xlabel('Rango de longitud (palabras)')
axes[1].set_ylabel('Número de muestras')
axes[1].set_title('Distribución de muestras por longitud')

plt.tight_layout()
plt.show()
print(acc_by_len.to_string())

Las gráficas sugieren que el rendimiento del modelo varía según la longitud del texto: en rangos con más ejemplos, la *precisión (accuracy)* se mantiene más estable y cercana al promedio global, mientras que en rangos con pocas muestras puede fluctuar más por efecto del tamaño de muestra. En general, no se observa una caída drástica del desempeño en textos largos, pero sí conviene interpretar con cuidado los extremos de longitud, donde hay menos datos y mayor incertidumbre.

## Inferencia en Texto Personalizado

Probamos el modelo con frases externas al dataset para evaluar su capacidad de generalización. Esto también sirve como demostración práctica del sistema completo de clasificación de sentimientos.

In [ ]:
import torch

def predecir_sentimiento(texto, seq_length=128):
    """Clasifica el sentimiento de un texto en español y devuelve las probabilidades."""
    model.eval()
    encoding = spanish_sentiment_tokenizer(
        texto,
        max_length=seq_length,
        truncation=True,
        padding='max_length',
        return_tensors='pt'
    )
    with torch.no_grad():
        logits = model(encoding['input_ids'], encoding['attention_mask'])
        probs  = torch.softmax(logits, dim=-1)[0]
        pred   = torch.argmax(probs).item()
    return {
        'texto':         texto,
        'sentimiento':   sentiment_dataset.id_2_class_map[pred],
        'prob_positivo': probs[1].item(),
        'prob_negativo': probs[0].item(),
    }

textos_prueba = [
    "El hotel fue increíble, muy limpio y el personal muy amable.",
    "Pésimo servicio, la habitación estaba sucia y el ruido era insoportable.",
    "La ubicación estaba bien pero el desayuno dejaba mucho que desear.",
    "Volveré sin duda, una experiencia maravillosa de principio a fin.",
    "No recomiendo este lugar para nada, fue una decepción total.",
    "El precio era razonable pero el servicio podría mejorar.",
]

print(f"{'Texto':<62} {'Sentimiento':<12} {'P(pos)':<8} {'P(neg)'}")
print('-' * 96)
for texto in textos_prueba:
    r = predecir_sentimiento(texto)
    print(f"{r['texto'][:60]:<62} {r['sentimiento'].upper():<12} {r['prob_positivo']:.2%}    {r['prob_negativo']:.2%}")

Las 6 frases de prueba fueron clasificadas como **positive**, incluso aquellas con polaridad claramente negativa (por ejemplo: *“Pésimo servicio...”* y *“No recomiendo este lugar...”*).

- Hay **sesgo fuerte hacia la clase positiva**.
- Las probabilidades están en un rango estrecho (~0.66–0.69), lo que sugiere **baja capacidad de separación** entre clases en inferencia libre.
- Esto es consistente con lo observado antes: dataset desbalanceado y dificultad en clase negativa.

El modelo parece haber aprendido un patrón de decisión casi constante hacia “positivo”, por lo que su utilidad práctica para textos ambiguos/negativos es limitada sin ajustes adicionales.

# 12. Conclusiones

En este notebook se implementó y evaluó un modelo basado en arquitectura Transformer para la clasificación de sentimientos en español, para comparar el desempeño con un modelo LSTM bidireccional.

### Arquitectura

#### Modelo LSTM

- Word embeddings para representar las palabras.

- LSTM bidireccional para capturar dependencias en ambas direcciones del texto.

- Capas densas finales para realizar la clasificación del sentimiento.

#### Modelo Transformer

- Tokenización de los textos para convertirlos en representaciones numéricas.

- Embeddings de tokens y posiciones para representar el contenido y el orden de las palabras.

- Bloque de Multi-Head Attention para capturar relaciones entre palabras dentro de la secuencia.

- Capas densas finales para predecir el sentimiento.

### Resultados

- El modelo LSTM alcanzó una precisión aproximada del 87%, mostrando un buen desempeño general en la clasificación.

- El análisis del reporte de clasificación muestra que el modelo es mucho más preciso identificando comentarios positivos que negativos, lo cual está relacionado con el desbalance de clases presente en el dataset.

- El modelo Transformer obtuvo una precisión cercana al 83% haciendo tratamiento en el balance de clases en el conjunto de prueba, demostrando que también es capaz de aprender patrones relevantes en los textos mediante el mecanismo de atención.

### Comparación entre modelos

- El LSTM logró un desempeño ligeramente superior en precisión general, posiblemente debido a que el dataset es relativamente pequeño.

- El Transformer tiene la ventaja de capturar relaciones globales entre palabras mediante el mecanismo de atención, lo que lo hace especialmente útil en textos más largos o datasets más grandes.

- Ambos modelos muestran buen desempeño en la clase positiva, pero presentan mayores dificultades para clasificar correctamente comentarios negativos.

### Posibles mejoras

- Aumentar el tamaño del dataset para mejorar la capacidad de generalización del modelo.

- Usar modelos preentrenados como BERT o BETO, que suelen ofrecer mejores resultados en tareas de NLP.

- Ajustar hiperparámetros como tamaño de embeddings, número de capas o número de cabezas de atención.

También se evidenció que el uso de modelos basados en arquitecturas de tipo Transformer resulta más provechoso cuando se dispone de datasets de mayor tamaño. Esto se debe a que estos modelos cuentan con un número considerable de parámetros y están diseñados para capturar relaciones complejas y dependencias contextuales dentro del texto, lo cual requiere una cantidad suficiente de datos para ser aprendido adecuadamente.

En conclusión, ambos enfoques muestran buenos resultados para la clasificación de sentimientos en español. Sin embargo, los modelos basados en Transformers ofrecen mayor flexibilidad y escalabilidad, especialmente cuando se trabaja con grandes volúmenes de datos o modelos preentrenados.